# 第2ポートフォリオ(P2) 検証 — 「ターン・オブ・ウィーク株価指数」バスケット

既存P1(v7円月曜＋v4合議FX＋E5多資産モメンタム)とは**別物・低相関**な第2ポートフォリオ(P2)を、
P1と**同一の v7基準 9ゲート**で採点する自己完結ノート。詳細は `docs/50_p2_turnofweek_portfolio.md`。

## 採用エッジ
**米独株価指数(US500 / NAS100 / GER40)の「月曜LONG」(週初ドリフト)バスケット**
= STRONG-LEAD(P1のv7/E5と同級)。**P1との月次相関 0.014(ほぼゼロ)** が最大の価値。

## 🚀 使い方（Colab）
1. （実データ確証する場合）Google Drive の `MyDrive/forex_ml/multiasset_daily/` に
   `US500_d.csv` / `NAS100_d.csv` / `GER40_d.csv`(列: timestamp,open,high,low,close, 日足10年)を置く。
   無ければ **Yahoo(^GSPC/^IXIC/^GDAXI)を自動取得**(研究用フォールバック)。
2. 「ランタイム → すべてのセルを実行」。
3. 第1セル=9ゲート採点(判定 STRONG-LEAD)、第2セル=独立インストゥルメント追認(現物/ETF/先物 8/8)。

## このノートが答える問い
- 月曜LONGは**月曜に特異か**(プラセボ: 火-金は非有意か)？
- IS/OOS両+ / 年次JK / ウォークフォワード / コスト2× を通るか？
- **P1と独立か**(相関≈0)？ −10%枠に収まるサイズは(チャレンジMC)？
- 現物だけでなく**ETF・先物でも再現**するか(Yahoo現物固有のクセでない傍証)？

> ⚠ シミュレーション。株価指数=配当抜きprice index。CFD実スプレッド/オーバーナイトスワップ/配当は
> 未計上＝**デモで実測**。最終確証はユーザーの実データ・デモ前進検証で(P1と同方針)。「必ず通る手法」は無い。


In [1]:
# 依存(Colabは標準で入っているが念のため)
!pip -q install numpy pandas >/dev/null 2>&1
print("ready")

ready


## 1. 本検証 — P1のv7基準 9ゲートで P2核を採点
`research/colab_p2_validate.py` と同一。Drive実指数があれば最優先、無ければYahoo自動取得。

In [2]:
# -*- coding: utf-8 -*-
"""
colab_p2_validate.py — 第2ポートフォリオ(P2)「ターン・オブ・ウィーク株価指数バスケット」を
                       ユーザーの実データ(Colab/Drive)で再確認するための自己完結スクリプト。

P1のv7基準と同一の9ゲートで P2核(US500/NAS100/GER40 月曜LONG)を採点する。
ローカル(research/p2_portfolio_validate.py)はYahoo日足の結論を、本スクリプトは
ユーザーのDrive実データ(あれば)で追認する。数字は盛らない——届かないものは届かないと出す。

データ優先順位:
  1) Drive: {DAILY_DIR}/{US500,NAS100,GER40}_d.csv (ユーザーの実指数日足があれば最優先)
  2) Yahoo自動取得(^GSPC, ^IXIC, ^GDAXI) フォールバック
列: timestamp, open, high, low, close (UTC, 日足)。

9ゲート(P1のv7基準):
  G1 10年 / G2 ノールックアヘッド+実コスト(構造) / G3 プールperm_p<Bonferroni(N=274,α=0.000182) /
  G4 プラセボ(月曜のみ有意・火-金非有意) / G5 年次JK max_p<=0.10 / G6 IS/OOS両+ /
  G7 WF>=4/5 / G8 コスト2×で net>0 / G9 −10%枠適合(ブロックBSのp95 maxDD)
判定: ADOPT=全通過 / STRONG-LEAD=G4,G6,G7,G8通過でG3/G5が未達 / LEAD=それ未満。

⚠ 指数=配当抜きprice index。CFD実スプレッド/オーバーナイトスワップ/配当はデモで実測。
   最終確証はデモ前進検証(docs/29と同方針)。シミュレーションは将来約定を保証しない。
"""
import os, json, urllib.request, datetime as dt
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

USE_DRIVE  = True
DRIVE_BASE = "/content/drive/MyDrive/forex_ml"
DAILY_DIR  = "{base}/multiasset_daily"
LOCAL_FALLBACK = "./research/data"
CORE   = ["US500","NAS100","GER40"]
YH     = {"US500":"^GSPC","NAS100":"^IXIC","GER40":"^GDAXI"}
BPS    = 5.0                  # 往復コスト(指数, ベーシスポイント)
N_SEARCH = 274               # ローカル探索 p2_edge_search.py の全試行数
BONF   = 0.05/N_SEARCH       # ≈0.000182
SEED   = 20260608

if USE_DRIVE:
    try:
        if not os.path.exists("/content/drive/MyDrive"):
            from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Drive不可(ローカル継続):", e)
DRIVE_OK = os.path.exists("/content/drive/MyDrive")

def _yahoo(sym):
    u=f"https://query2.finance.yahoo.com/v8/finance/chart/{sym}?interval=1d&range=10y"
    req=urllib.request.Request(u, headers={"User-Agent":"Mozilla/5.0"})
    d=json.loads(urllib.request.urlopen(req, timeout=25).read())
    r=d["chart"]["result"][0]; ts=r["timestamp"]; q=r["indicators"]["quote"][0]
    rows=[]
    for i,t in enumerate(ts):
        o,h,l,c=q["open"][i],q["high"][i],q["low"][i],q["close"][i]
        if None in (o,h,l,c): continue
        rows.append((dt.datetime.utcfromtimestamp(t).strftime("%Y-%m-%d %H:%M:%S"),o,h,l,c))
    return pd.DataFrame(rows, columns=["timestamp","open","high","low","close"])

def load(name):
    cands=[]
    if DRIVE_OK: cands.append(f"{DAILY_DIR.format(base=DRIVE_BASE)}/{name}_d.csv")
    cands.append(f"{LOCAL_FALLBACK}/{name}_d.csv")
    df=None
    for p in cands:
        if os.path.exists(p): df=pd.read_csv(p); print(f"  {name}: ローカル/Drive {p}"); break
    if df is None:
        print(f"  {name}: Yahoo {YH[name]} を取得"); df=_yahoo(YH[name])
    df["t"]=pd.to_datetime(df["timestamp"],utc=True,errors="coerce")
    df=df.dropna(subset=["t"]).sort_values("t")
    df["trade_date"]=(df["t"]+pd.Timedelta(hours=2)).dt.floor("D")
    df=df.groupby("trade_date").last()
    df["weekday"]=df.index.dayofweek
    df=df[df["weekday"]<=4]
    df["o2o"]=df["open"].shift(-1)/df["open"]-1.0
    return df.dropna(subset=["o2o"])

def perm_p(x,n=5000,seed=11):
    x=np.asarray(x,float)
    if len(x)<10: return 1.0
    rng=np.random.default_rng(seed); real=x.sum(); a=np.abs(x).astype(np.float32)
    null=(rng.choice(np.array([-1,1],np.float32),size=(n,len(a)))*a).sum(axis=1)
    return float((null>=real).mean())

def eqstats(x):
    x=pd.Series(x).dropna();
    if len(x)==0: return dict(net_pct=0,maxDD_pct=0,n=0,win_pct=0)
    eq=(1+x).cumprod(); dd=((eq-eq.cummax())/eq.cummax()).min()*100
    return dict(net_pct=round((eq.iloc[-1]-1)*100,1),maxDD_pct=round(float(dd),1),
                n=int(len(x)),win_pct=round(float((x>0).mean())*100,1))

def basket(weekdays, mult=1.0):
    parts=[]
    for nm in CORE:
        df=DF[nm]; sub=df[df["weekday"].isin(weekdays)]
        r=(sub["o2o"]-mult*BPS/1e4); r.index=r.index.to_period("W")
        parts.append(r.groupby(r.index).mean().rename(nm))
    return pd.concat(parts,axis=1).mean(axis=1).dropna()

def mc_p95dd(weekly, L, n_paths=4000, block=4, maxw=520, target=0.08, floor=-0.10, seed=SEED):
    w=np.asarray(weekly)*L; n=len(w); rng=np.random.default_rng(seed); mdds=[]; npass=ndq=0
    for _ in range(n_paths):
        eq=1.0; peak=1.0; mdd=0.0; k=0
        while k<maxw:
            st=rng.integers(0,max(1,n-block))
            for r in w[st:st+block]:
                eq*=(1+r); peak=max(peak,eq); mdd=min(mdd,eq/peak-1.0); k+=1
                if eq-1<=floor: ndq+=1; mdd=mdd; break
                if eq-1>=target: npass+=1; break
            else: continue
            break
        mdds.append(mdd)
    return dict(L=L,p95_maxDD=round(float(np.percentile(mdds,5))*100,1),
                pass_pct=round(100*npass/n_paths,1),dq_pct=round(100*ndq/n_paths,1))

print("="*72); print("P2 検証(実データ確認): ターン・オブ・ウィーク株価指数バスケット", CORE); print("="*72)
DF={nm:load(nm) for nm in CORE}
span=DF["US500"].index
print(f"\n期間: {span.min().date()} .. {span.max().date()}  (US500 {len(DF['US500'])}本)")

# 個別 月曜LONG + 曜日プラセボ
print("\n[個別指数 月曜LONG / 曜日プラセボ]")
for nm in CORE:
    for w in range(5):
        sub=DF[nm][DF[nm]["weekday"]==w]; r=(sub["o2o"]-BPS/1e4)
        tag="★Mon" if w==0 else ["Mon","Tue","Wed","Thu","Fri"][w]
        if w==0 or w==4:
            print(f"  {nm} {['Mon','Tue','Wed','Thu','Fri'][w]}: net{eqstats(r.values)['net_pct']:6.1f}% p={perm_p(r.values):.4f}")

Bmon=basket([0]); Both=basket([1,2,3,4])
p_mon=perm_p(Bmon.values); p_oth=perm_p(Both.values)
half=Bmon.index[len(Bmon)//2]; Ris=Bmon[Bmon.index<half]; Roos=Bmon[Bmon.index>=half]
yrs=sorted(set(Bmon.index.year)); jk={int(y):round(perm_p(Bmon[Bmon.index.year!=y].values),3) for y in yrs}
jkmax=max(jk.values())
wf=sum(1 for s in range(5) if eqstats(Bmon[(Bmon.index>=Bmon.index[int(len(Bmon)*s/5)])&(Bmon.index< (Bmon.index[min(len(Bmon)-1,int(len(Bmon)*(s+1)/5))]))].values)["net_pct"]>0)
B2=basket([0],2.0)
g={"G3":p_mon<BONF,"G4":(p_mon<0.05 and p_oth>0.05),"G5":jkmax<=0.10,
   "G6":(Ris.sum()>0 and Roos.sum()>0),"G7":wf>=4,"G8":eqstats(B2.values)["net_pct"]>0}
mc_sweep=[mc_p95dd(Bmon.values,L) for L in (0.5,0.6,0.75,1.0)]
mc=mc_sweep[0]   # 推奨保守サイズ L=0.5 で G9 判定
g["G9"]=mc["p95_maxDD"]>=-10.0

print(f"\n[G3/G4] 月曜プールnet{eqstats(Bmon.values)['net_pct']}% perm_p={p_mon:.4f}(Bonf α={BONF:.5f}={'OK' if g['G3'] else 'NG'}) / "
      f"火-金プラセボ net{eqstats(Both.values)['net_pct']}% p={p_oth:.4f}({'識別OK' if g['G4'] else 'NG'})")
print(f"[G6] IS net{eqstats(Ris.values)['net_pct']}% / OOS net{eqstats(Roos.values)['net_pct']}% → {g['G6']}")
print(f"[G5] 年次JK max_p={jkmax:.3f} → {g['G5']}  {jk}")
print(f"[G7] WF {wf}/5 → {g['G7']}    [G8] コスト2× net{eqstats(B2.values)['net_pct']}% → {g['G8']}")
print(f"[G9] ブロックBS レバ感度: " + " / ".join(f"L{m['L']}:p95DD{m['p95_maxDD']}% 合格{m['pass_pct']}%" for m in mc_sweep))
print(f"     推奨保守L=0.5 → p95 maxDD {mc['p95_maxDD']}% 失格{mc['dq_pct']}% → {g['G9']} (−10%枠の内側)")

npass=sum(g.values())
adopt=all(g.values())
grade="ADOPT" if adopt else ("STRONG-LEAD" if (g["G4"] and g["G6"] and g["G7"] and g["G8"]) else "LEAD")
print(f"\n→ {npass}/7コア+G9 / 判定: {grade}")
if grade!="ADOPT":
    miss=[k for k,v in g.items() if not v]
    print(f"   未達: {miss}  ＝P1のv7/E5と同じ『質は高いが最厳の統計閾値のみ未達』。確証はデモ前進検証で。")
print("\n免責: シミュレーション。指数=配当抜き。CFD実コスト/スワップ/配当はデモで実測。将来約定を保証しない。")


Drive不可(ローカル継続): No module named 'google'
P2 検証(実データ確認): ターン・オブ・ウィーク株価指数バスケット ['US500', 'NAS100', 'GER40']
  US500: Yahoo ^GSPC を取得


  NAS100: Yahoo ^IXIC を取得


  GER40: Yahoo ^GDAXI を取得

期間: 2016-06-06 .. 2026-06-04  (US500 2514本)

[個別指数 月曜LONG / 曜日プラセボ]


  US500 Mon: net  51.9% p=0.0194
  US500 Fri: net -14.4% p=0.6948
  NAS100 Mon: net 117.8% p=0.0028
  NAS100 Fri: net -23.5% p=0.7398
  GER40 Mon: net  65.5% p=0.0202
  GER40 Fri: net -17.2% p=0.6732



[G3/G4] 月曜プールnet68.7% perm_p=0.0098(Bonf α=0.00018=NG) / 火-金プラセボ net-9.2% p=0.7540(識別OK)
[G6] IS net38.4% / OOS net21.9% → True
[G5] 年次JK max_p=0.234 → False  {2016: 0.01, 2017: 0.011, 2018: 0.0, 2019: 0.008, 2020: 0.234, 2021: 0.007, 2022: 0.002, 2023: 0.01, 2024: 0.008, 2025: 0.012, 2026: 0.013}
[G7] WF 4/5 → True    [G8] コスト2× net30.4% → True
[G9] ブロックBS レバ感度: L0.5:p95DD-9.6% 合格96.6% / L0.6:p95DD-10.8% 合格96.0% / L0.75:p95DD-12.0% 合格93.5% / L1.0:p95DD-13.2% 合格89.2%
     推奨保守L=0.5 → p95 maxDD -9.6% 失格1.6% → True (−10%枠の内側)

→ 5/7コア+G9 / 判定: STRONG-LEAD
   未達: ['G3', 'G5']  ＝P1のv7/E5と同じ『質は高いが最厳の統計閾値のみ未達』。確証はデモ前進検証で。

免責: シミュレーション。指数=配当抜き。CFD実コスト/スワップ/配当はデモで実測。将来約定を保証しない。


## 2. 独立インストゥルメント追認（現物 / ETF / 先物）
同じ指数を ETF(SPY/QQQ/EWG)・先物(ES/NQ) で再検定。月曜のみ有意が再現すれば
「Yahoo現物固有のクセ」でなく本物の市場現象の傍証。先物再現はCFD実運用妥当性も補強。
`research/p2_confirm_proxies.py` と同一。

In [3]:
# -*- coding: utf-8 -*-
"""
p2_confirm_proxies.py — P2「株価指数 月曜LONG」エッジの独立インストゥルメント追認。

目的: ユーザーのDrive実データに直接アクセスできない環境での追認として、同じ指数を
  【別インストゥルメント=ETF・先物】で取得し、月曜エッジが「Yahoo現物指数データ固有の
  クセ」ではなく、現物/ETF/先物を通じて再現する本物の現象かを交差確認する。
  CFDは先物/現物に連動するため、ETF・先物での再現は実運用妥当性の追認にもなる。

確認対象(全てYahoo・10年日足だが、現物とは別系列のインストゥルメント):
  US株: 現物^GSPC / ETF SPY / 先物 ES=F   ・ Nasdaq: 現物^IXIC / ETF QQQ / 先物 NQ=F
  独株: 現物^GDAXI / ETF EWG(独株ETF)
判定: 各々で「月曜LONGのperm_p」と「火-金プラセボ」を出し、月曜のみ有意が再現するか。
注: ETF/先物は配当・限月・乖離があり現物と完全一致しない＝独立性の担保。最終確証はユーザー実CFDデモで。
"""
import os, json, urllib.request, datetime as dt
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

try:
    HERE=os.path.dirname(os.path.abspath(__file__))   # スクリプト実行時
except NameError:
    HERE=os.getcwd()                                   # ノートブックcell実行時(__file__無し)
BPS=5.0
GROUPS={
 "US500":  [("現物 ^GSPC","^GSPC"),("ETF SPY","SPY"),("先物 ES=F","ES=F")],
 "NAS100": [("現物 ^IXIC","^IXIC"),("ETF QQQ","QQQ"),("先物 NQ=F","NQ=F")],
 "GER40":  [("現物 ^GDAXI","^GDAXI"),("ETF EWG","EWG")],
}

def yahoo(sym):
    u=f"https://query2.finance.yahoo.com/v8/finance/chart/{sym}?interval=1d&range=10y"
    req=urllib.request.Request(u, headers={"User-Agent":"Mozilla/5.0"})
    d=json.loads(urllib.request.urlopen(req, timeout=25).read())
    r=d["chart"]["result"][0]; ts=r["timestamp"]; q=r["indicators"]["quote"][0]
    rows=[(dt.datetime.utcfromtimestamp(t), q["open"][i],q["close"][i])
          for i,t in enumerate(ts) if None not in (q["open"][i],q["close"][i])]
    df=pd.DataFrame(rows,columns=["t","open","close"])
    df["t"]=pd.to_datetime(df["t"],utc=True)
    df["trade_date"]=(df["t"]+pd.Timedelta(hours=2)).dt.floor("D")
    df=df.groupby("trade_date").last(); df["weekday"]=df.index.dayofweek
    df=df[df["weekday"]<=4]; df["o2o"]=df["open"].shift(-1)/df["open"]-1.0
    return df.dropna(subset=["o2o"])

def perm_p(x,n=5000,seed=11):
    x=np.asarray(x,float)
    if len(x)<10: return 1.0
    rng=np.random.default_rng(seed); real=x.sum(); a=np.abs(x).astype(np.float32)
    return float(((rng.choice(np.array([-1,1],np.float32),size=(n,len(a)))*a).sum(axis=1)>=real).mean())

def net(x):
    x=pd.Series(x).dropna(); return round(((1+x).cumprod().iloc[-1]-1)*100,1) if len(x) else 0.0

def mon_other(df):
    mon=df[df["weekday"]==0]["o2o"]-BPS/1e4
    oth=df[df["weekday"].isin([1,2,3,4])]["o2o"]-BPS/1e4
    return (net(mon.values),perm_p(mon.values),len(mon)),(net(oth.values),perm_p(oth.values))

def main():
    print("="*74); print("P2 月曜エッジ 独立インストゥルメント追認（現物 / ETF / 先物）"); print("="*74)
    out={}; n_mon_sig=0; n_total=0
    for grp,members in GROUPS.items():
        print(f"\n[{grp}]")
        out[grp]={}
        for label,sym in members:
            try:
                df=yahoo(sym); (mn,mp,nn),(on,op)=mon_other(df)
                span=f"{df.index.min().date()}..{df.index.max().date()}"
                sig = mp<=0.05; plac_ok = op>0.05
                verdict = "✅月曜のみ有意" if (sig and plac_ok) else ("△月曜有意/プラセボも" if sig else "✗月曜非有意")
                print(f"  {label:11s} 月曜 net{mn:6.1f}% p={mp:.4f} (n{nn}) | 火-金 net{on:6.1f}% p={op:.3f} → {verdict}  [{span}]")
                out[grp][sym]=dict(mon_net=mn,mon_p=round(mp,4),mon_n=nn,other_net=on,other_p=round(op,3),span=span)
                n_total+=1; n_mon_sig+= 1 if sig else 0
            except Exception as e:
                print(f"  {label:11s} ERR {type(e).__name__} {str(e)[:50]}")
    print(f"\n=== 追認サマリ: 月曜LONGが p<=0.05 で再現したインストゥルメント = {n_mon_sig}/{n_total} ===")
    print("→ 現物だけでなくETF/先物でも月曜効果が再現すれば『Yahoo現物固有のクセ』ではない＝本物の現象の傍証。")
    print("  (絶対値はインストゥルメントで異なる＝配当/限月/乖離。最終確証はユーザー実CFDデモで)")
    os.makedirs(os.path.join(HERE,"results"),exist_ok=True)
    with open(os.path.join(HERE,"results","p2_confirm_proxies.json"),"w") as f:
        json.dump(dict(summary=f"{n_mon_sig}/{n_total}",groups=out),f,ensure_ascii=False,indent=2)
    print("\n保存: results/p2_confirm_proxies.json")

if __name__=="__main__":
    main()


P2 月曜エッジ 独立インストゥルメント追認（現物 / ETF / 先物）

[US500]


  現物 ^GSPC    月曜 net  51.9% p=0.0194 (n469) | 火-金 net -32.8% p=0.740 → ✅月曜のみ有意  [2016-06-06..2026-06-04]


  ETF SPY     月曜 net  80.8% p=0.0038 (n469) | 火-金 net -43.8% p=0.819 → ✅月曜のみ有意  [2016-06-06..2026-06-04]


  先物 ES=F     月曜 net  50.3% p=0.0402 (n467) | 火-金 net -33.9% p=0.707 → ✅月曜のみ有意  [2016-06-07..2026-06-05]

[NAS100]


  現物 ^IXIC    月曜 net 117.8% p=0.0028 (n469) | 火-金 net -29.9% p=0.608 → ✅月曜のみ有意  [2016-06-06..2026-06-04]


  ETF QQQ     月曜 net 150.0% p=0.0000 (n469) | 火-金 net -24.6% p=0.564 → ✅月曜のみ有意  [2016-06-06..2026-06-04]


  先物 NQ=F     月曜 net  93.9% p=0.0094 (n467) | 火-金 net  -6.5% p=0.419 → ✅月曜のみ有意  [2016-06-07..2026-06-05]

[GER40]


  現物 ^GDAXI   月曜 net  65.5% p=0.0202 (n496) | 火-金 net -58.1% p=0.908 → ✅月曜のみ有意  [2016-06-06..2026-06-04]


  ETF EWG     月曜 net  75.1% p=0.0156 (n469) | 火-金 net -73.3% p=0.969 → ✅月曜のみ有意  [2016-06-06..2026-06-04]

=== 追認サマリ: 月曜LONGが p<=0.05 で再現したインストゥルメント = 8/8 ===
→ 現物だけでなくETF/先物でも月曜効果が再現すれば『Yahoo現物固有のクセ』ではない＝本物の現象の傍証。
  (絶対値はインストゥルメントで異なる＝配当/限月/乖離。最終確証はユーザー実CFDデモで)

保存: results/p2_confirm_proxies.json
